# ML Model Training and Evaluation

This notebook trains multiple models, compares their performance, selects the best model, and saves the final model artifacts.

In [ ]:
# Importing required libraries
import os
import re
import pickle
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Loading the dataset
DATASET_PATH = '../../milestone_3_ml/dataset/training_data.csv'
df = pd.read_csv(DATASET_PATH)
df.head()


In [ ]:
# Preparing the input text
df = df.dropna(subset=['title', 'description', 'priority']).copy()
df['text'] = df['title'] + ' ' + df['description']
df[['text', 'priority']].head()


In [ ]:
# Cleaning the text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()


In [ ]:
# Separating input and target
X = df['clean_text']
y = df['priority']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training records:', len(X_train))
print('Testing records:', len(X_test))


In [ ]:
# Converting text into TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print('Training feature shape:', X_train_vec.shape)
print('Testing feature shape:', X_test_vec.shape)


## Model Training

In [ ]:
# Training Multinomial Naive Bayes
naive_bayes = MultinomialNB()
naive_bayes.fit(X_train_vec, y_train)


In [ ]:
# Training Logistic Regression
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)
logistic_regression.fit(X_train_vec, y_train)


In [ ]:
# Training Linear SVM
linear_svm = SVC(kernel='linear', probability=True, random_state=42)
linear_svm.fit(X_train_vec, y_train)


## Model Evaluation

In [ ]:
# Comparing model performance
models = {
    'Naive Bayes': naive_bayes,
    'Logistic Regression': logistic_regression,
    'Linear SVM': linear_svm
}

results = []

for name, model in models.items():
    predictions = model.predict(X_test_vec)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, average='macro', zero_division=0),
        'Recall': recall_score(y_test, predictions, average='macro', zero_division=0),
        'F1 Score': f1_score(y_test, predictions, average='macro', zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Plotting model comparison
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1 Score']].plot(kind='bar', figsize=(10, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Selecting the best model
best_model_name = results_df.loc[0, 'Model']
best_model = models[best_model_name]

print('Best model:', best_model_name)
print('Best F1 Score:', round(results_df.loc[0, 'F1 Score'], 4))


In [ ]:
# Showing the best model classification report
best_predictions = best_model.predict(X_test_vec)
print(classification_report(y_test, best_predictions, zero_division=0))


In [ ]:
# Showing the best model confusion matrix
labels = sorted(y.unique())
cm = confusion_matrix(y_test, best_predictions, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(xticks_rotation=45)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()


## Saving the Final Model

In [ ]:
# Creating the artifacts folder
ARTIFACTS_PATH = '../artifacts'
os.makedirs(ARTIFACTS_PATH, exist_ok=True)


In [ ]:
# Saving the best model
model_path = os.path.join(ARTIFACTS_PATH, 'model.pkl')
vectorizer_path = os.path.join(ARTIFACTS_PATH, 'vectorizer.pkl')

with open(model_path, 'wb') as file:
    pickle.dump(best_model, file)

with open(vectorizer_path, 'wb') as file:
    pickle.dump(vectorizer, file)

print('Model saved to:', model_path)
print('Vectorizer saved to:', vectorizer_path)


In [ ]:
# Checking the saved artifacts
print('Model file exists:', os.path.exists(model_path))
print('Vectorizer file exists:', os.path.exists(vectorizer_path))
